# CV4CHL - Model Diversity Item 12

**Platform**: Colab / Local (CPU only)
**Runtime**: ~10-20 min
**Goal**: Use LOSO CV to systematically search item-source combinations

## Strategy

We have evgs_rules's 17 per-item tuned rule predictions (OOF from LOSO CV).
Use leave-one-subject-out CV to evaluate item-source combinations:
1. Build all possible combos in CV space
2. Use greedy forward selection from a LOSO-validated base (items 6,13,17 selected
by per-item train accuracy)
3. Export top candidate source combinations for downstream assembly

## Validation protocol

Leave-one-subject-out CV evaluates each candidate item-source combination
entirely on training data. Model selection is therefore decided by
training-set generalization, not by held-out test performance.


In [ ]:
# === Cell 1: Setup & Imports ===
import time, os, sys, pickle, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

_cell_times = {}
_cell_start = None
def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'\u25b6 {name}')
def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'\u2713 {name} — {elapsed:.1f}s ({elapsed/60:.1f}min)')

try:
    import xgboost as xgb
except ImportError:
    os.system('pip install -q xgboost'); import xgboost as xgb
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip install -q lightgbm'); import lightgbm as lgb
try:
    from catboost import CatBoostClassifier
except ImportError:
    os.system('pip install -q catboost'); from catboost import CatBoostClassifier

print('Setup complete.')


In [ ]:
# === Cell 2: Config & Load Data ===
cell_start('Cell 2: Config & Load Data')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        '/content/drive/MyDrive/CVPR_CH4CHL',
    )
else:
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        os.getcwd(),
    )

PROC_DIR = f'{ROOT}/1_data/processed'
SUBMIT_DIR = f'{ROOT}/5_outputs/submissions'
DIAGNOSTIC_DIR = f'{ROOT}/5_outputs/diagnostics/model_diversity_item12'
os.makedirs(SUBMIT_DIR, exist_ok=True)
os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)

T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]
# Track-2 rows are preserved from the current base CSV;
# Track-2 predictions are generated only by the Track-2 clinical notebook.
EVGS_ITEMS = [f'evgs_{i}' for i in range(1, 18)]

# Load preprocessed features + labels
with open(f'{PROC_DIR}/train_ready.pkl', 'rb') as f:
    tr_data = pickle.load(f)
df_train = tr_data['track1_train'].copy()
df_test = tr_data['track1_test'].copy()

FEAT_COLS = sorted([c for c in df_train.columns
    if c.startswith(('sag_', 'cor_', 'all_', 'n_'))])

print(f'Train: {df_train.shape}, Test: {df_test.shape}')
print(f'Features: {len(FEAT_COLS)}')

cell_end('Cell 2: Config & Load Data')


In [ ]:
# === Cell 3: Build Per-Item OOF Predictions (ML baseline + evgs_rules rules) ===
cell_start('Cell 3: Build OOF Predictions')

train_pids = sorted(df_train['patient_id'].unique())
n_train = len(df_train)

# ==============================
# ML baseline: per-item LOSO CV with best model from reference baseline pipeline
# Models: XGBoost default, plus CatBoost/LightGBM for specific items
# ==============================

ITEM_MODELS = {
    # Per-item best models from LOSO-CV-validated model-diversity selection
    5: 'CatBoost', 6: 'LightGBM', 7: 'CatBoost',
    8: 'CatBoost', 10: 'CatBoost', 13: 'CatBoost',
}

ml_oof = {}  # {item_i: np.array of 0/1 predictions}

for item_i in tqdm(range(1, 18), desc='ML OOF'):
    item_name = f'evgs_{item_i}'
    y = df_train[item_name].values
    X = np.nan_to_num(df_train[FEAT_COLS].values.astype(np.float32), nan=0.0)
    oof = np.zeros(n_train, dtype=np.float32)
    model_type = ITEM_MODELS.get(item_i, 'XGBoost')

    for pid in train_pids:
        mask = df_train['patient_id'].values == pid
        X_tr, y_tr = X[~mask], y[~mask]
        X_val = X[mask]
        spw = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)

        if model_type == 'XGBoost':
            clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                scale_pos_weight=spw, use_label_encoder=False,
                eval_metric='logloss', verbosity=0, random_state=42)
        elif model_type == 'LightGBM':
            clf = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                scale_pos_weight=spw, verbose=-1, random_state=42)
        else:
            clf = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
                auto_class_weights='Balanced', verbose=0, random_state=42)
        clf.fit(X_tr, y_tr)
        oof[mask] = clf.predict(X_val)
    ml_oof[item_i] = oof

# ==============================
# Model-diversity selection works in OOF space: the ML LOSO OOF
# predictions computed above feed the per-item model-diversity search
# below, which evaluates per-item model swaps in the same OOF space.
# ==============================

print(f'\nML OOF computed for {len(ml_oof)} items')

# Per-item ML accuracy
print(f'\n{"Item":>4} {"ML_Acc":>8} {"PositiveRate":>12}')
for i in range(1, 18):
    y = df_train[f'evgs_{i}'].values
    acc = accuracy_score(y, ml_oof[i])
    pos = y.mean()
    print(f'{i:>4d} {acc:>8.4f} {pos:>12.3f}')

cell_end('Cell 3: Build OOF Predictions')


In [ ]:
# === Cell 4: Compute S1 for Any Combo ===
cell_start('Cell 4: S1 Computation Engine')

def compute_s1_from_oof(oof_dict, items_to_overlay=None, overlay_oof=None):
    """Compute S1 from OOF predictions.

    oof_dict: {item_i: np.array} — base predictions (ML)
    items_to_overlay: set of item indices to replace with overlay predictions
    overlay_oof: {item_i: np.array} — alternative predictions for overridden items

    Returns: (acc, rmse, s1)
    """
    if items_to_overlay is None:
        items_to_overlay = set()
    if overlay_oof is None:
        overlay_oof = {}

    n = len(df_train)
    all_correct = 0
    all_total = 0
    total_pred = np.zeros(n)
    total_true = np.zeros(n)

    for item_i in range(1, 18):
        if item_i in items_to_overlay and item_i in overlay_oof:
            pred = overlay_oof[item_i]
        else:
            pred = oof_dict[item_i]
        true = df_train[f'evgs_{item_i}'].values
        all_correct += np.sum(pred == true)
        all_total += len(true)
        total_pred += pred
        total_true += true

    acc = all_correct / all_total

    # RMSE per-patient (L+R combined Total)
    pids = df_train['patient_id'].values
    sides = df_train['side'].values
    unique_pids = sorted(set(pids))
    rmse_sq = []
    for pid in unique_pids:
        ml = (pids == pid) & (sides == 'left')
        mr = (pids == pid) & (sides == 'right')
        if ml.sum() > 0 and mr.sum() > 0:
            pt = total_pred[ml][0] + total_pred[mr][0]
            tt = total_true[ml][0] + total_true[mr][0]
            rmse_sq.append((pt - tt) ** 2)
    rmse = np.sqrt(np.mean(rmse_sq)) if rmse_sq else 0.0
    s1 = (acc + 1 - rmse / 34.0) / 2
    return acc, rmse, s1

# Baseline S1 (all ML)
_, _, s1_baseline = compute_s1_from_oof(ml_oof)
print(f'Baseline ML S1: {s1_baseline:.5f}')

cell_end('Cell 4: S1 Computation Engine')


In [ ]:
# === Cell 5: Generate Alternative OOF (Flipped Predictions) ===
cell_start('Cell 5: Alternative OOFs')

# Build per-item alternative predictions for the model-diversity search:
# for each EVGS item, train with each of the 3 base models (XGBoost,
# LightGBM, CatBoost) and compare per-item LOSO OOF performance. The
# model-diversity selection below picks which model to use per item.

alt_oof = {}  # {(item_i, model_name): np.array}

MODELS = ['XGBoost', 'LightGBM', 'CatBoost']

for item_i in tqdm(range(1, 18), desc='Alt OOF'):
    item_name = f'evgs_{item_i}'
    y = df_train[item_name].values
    X = np.nan_to_num(df_train[FEAT_COLS].values.astype(np.float32), nan=0.0)

    for model_name in MODELS:
        oof = np.zeros(n_train, dtype=np.float32)
        for pid in train_pids:
            mask = df_train['patient_id'].values == pid
            X_tr, y_tr = X[~mask], y[~mask]
            X_val = X[mask]
            spw = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)

            if model_name == 'XGBoost':
                clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                    scale_pos_weight=spw, use_label_encoder=False,
                    eval_metric='logloss', verbosity=0, random_state=42)
            elif model_name == 'LightGBM':
                clf = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                    scale_pos_weight=spw, verbose=-1, random_state=42)
            else:
                clf = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
                    auto_class_weights='Balanced', verbose=0, random_state=42)
            clf.fit(X_tr, y_tr)
            oof[mask] = clf.predict(X_val)
        alt_oof[(item_i, model_name)] = oof

print(f'Alt OOFs: {len(alt_oof)} (17 items x 3 models)')

# Per-item best model in CV
print(f'\n{"Item":>4} {"XGB":>8} {"LGB":>8} {"CB":>8} {"Best":>12} {"Delta":>8}')
print('-' * 55)
for i in range(1, 18):
    y = df_train[f'evgs_{i}'].values
    accs = {}
    for m in MODELS:
        accs[m] = accuracy_score(y, alt_oof[(i, m)])
    best_m = max(accs, key=accs.get)
    delta = accs[best_m] - accs['XGBoost']
    print(f'{i:>4d} {accs["XGBoost"]:>8.4f} {accs["LightGBM"]:>8.4f} {accs["CatBoost"]:>8.4f} {best_m:>12} {delta:>+8.4f}')

cell_end('Cell 5: Alternative OOFs')


In [ ]:
# === Cell 6: Greedy Forward Selection ===
cell_start('Cell 6: Greedy Model Diversity Selection')

# Greedy forward selection: starting from the all-XGBoost baseline,
# at each step switch one item to the per-item best alternative model
# (LightGBM or CatBoost) and keep the switch if S1 improves on LOSO CV.

# Step 1: Find per-item best model
best_model_per_item = {}
for i in range(1, 18):
    y = df_train[f'evgs_{i}'].values
    best_m = 'XGBoost'
    best_acc = accuracy_score(y, alt_oof[(i, 'XGBoost')])
    for m in ['LightGBM', 'CatBoost']:
        acc = accuracy_score(y, alt_oof[(i, m)])
        if acc > best_acc:
            best_acc = acc
            best_m = m
    best_model_per_item[i] = best_m

print('Per-item best model:')
for i, m in sorted(best_model_per_item.items()):
    marker = ' *' if m != 'XGBoost' else ''
    print(f'  Item {i}: {m}{marker}')

# Step 2: Greedy forward selection
# Start: all XGBoost
current_switches = set()
current_s1 = s1_baseline

print(f'\nGreedy forward selection (baseline S1={s1_baseline:.5f}):')
print(f'{"Step":>4} {"Added":>8} {"Model":>10} {"S1":>10} {"Delta":>10} {"Total switches"}')
print('-' * 65)

for step in range(17):
    best_add = None
    best_s1 = current_s1
    best_model = None

    for i in range(1, 18):
        if i in current_switches:
            continue
        for m in MODELS:
            if m == 'XGBoost' and i not in current_switches:
                continue  # already using XGBoost
            trial = current_switches | {i}
            overlay = {}
            for j in trial:
                overlay[j] = alt_oof[(j, best_model_per_item[j] if j != i else m)]
            # For items already switched, use their best model
            for j in current_switches:
                overlay[j] = alt_oof[(j, best_model_per_item[j])]
            overlay[i] = alt_oof[(i, m)]

            _, _, s1 = compute_s1_from_oof(ml_oof, trial, overlay)
            if s1 > best_s1:
                best_s1 = s1
                best_add = i
                best_model = m

    if best_add is None or best_s1 <= current_s1:
        print(f'  No improvement found. Stopping.')
        break

    current_switches.add(best_add)
    best_model_per_item[best_add] = best_model
    delta = best_s1 - current_s1
    current_s1 = best_s1
    print(f'{step+1:>4d} {best_add:>8d} {best_model:>10s} {current_s1:>10.5f} {delta:>+10.5f} {sorted(current_switches)}')

print(f'\nFinal greedy result: S1={current_s1:.5f} (delta={current_s1 - s1_baseline:+.5f})')
print(f'Items switched: {sorted(current_switches)}')
print(f'Models: {dict((i, best_model_per_item[i]) for i in sorted(current_switches))}')

# Step 3: Also try exhaustive search for small combos (up to 5 items)
print(f'\n{"="*60}')
print(f'Exhaustive search: top combos by size')
print(f'{"="*60}')

top_combos = []  # (s1, items, models)

# 1-item switches
for i in range(1, 18):
    for m in ['LightGBM', 'CatBoost']:
        overlay = {i: alt_oof[(i, m)]}
        _, _, s1 = compute_s1_from_oof(ml_oof, {i}, overlay)
        if s1 > s1_baseline:
            top_combos.append((s1, (i,), {i: m}))

# 2-item switches
for i, j in combinations(range(1, 18), 2):
    for mi in ['LightGBM', 'CatBoost']:
        for mj in ['LightGBM', 'CatBoost']:
            overlay = {i: alt_oof[(i, mi)], j: alt_oof[(j, mj)]}
            _, _, s1 = compute_s1_from_oof(ml_oof, {i, j}, overlay)
            if s1 > s1_baseline + 0.005:
                top_combos.append((s1, (i, j), {i: mi, j: mj}))

# 3-item switches
for combo in combinations(range(1, 18), 3):
    # Only try best model per item (not all combos — too many)
    overlay = {}
    items = set()
    for i in combo:
        bm = max(MODELS, key=lambda m: accuracy_score(df_train[f'evgs_{i}'].values, alt_oof[(i, m)]))
        if bm != 'XGBoost':
            overlay[i] = alt_oof[(i, bm)]
            items.add(i)
    if items:
        _, _, s1 = compute_s1_from_oof(ml_oof, items, overlay)
        if s1 > s1_baseline + 0.005:
            top_combos.append((s1, tuple(sorted(items)), {i: bm for i, bm in zip(items, [max(MODELS, key=lambda m: accuracy_score(df_train[f'evgs_{i}'].values, alt_oof[(i, m)])) for i in items])}))

# Sort and print top-20
top_combos.sort(key=lambda x: -x[0])
seen = set()
unique_top = []
for s1, items, models in top_combos:
    key = tuple(sorted(items))
    if key not in seen:
        seen.add(key)
        unique_top.append((s1, items, models))
    if len(unique_top) >= 20:
        break

print(f'\nTop 20 combos by CV S1:')
print(f'{"Rank":>4} {"S1":>10} {"Delta":>10} {"Items":>30} {"Models":>30}')
print('-' * 90)
for rank, (s1, items, models) in enumerate(unique_top[:20], 1):
    delta = s1 - s1_baseline
    items_str = str(list(items))
    models_str = str({i: models.get(i, '?') for i in items})
    print(f'{rank:>4d} {s1:>10.5f} {delta:>+10.5f} {items_str:>30} {models_str:>30}')

cell_end('Cell 6: Greedy Model Diversity Selection')


In [ ]:
# === Cell 7: Generate Top-N Submission CSVs ===
cell_start('Cell 7: Generate Submissions')

# Load base CSV
BASE_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'
if not os.path.exists(BASE_PATH):
    # Try reference baseline
    BASE_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'
if not os.path.exists(BASE_PATH):
    print(f'ERROR: No base CSV found!')
    df_base = None
else:
    df_base = pd.read_csv(BASE_PATH)
    print(f'Base: {BASE_PATH}')

if df_base is not None:
    t1_mask = df_base['ID'].str.startswith('track1-')
    item_cols_csv = [f'{p}{i}' for p in ['L','R'] for i in range(1,18)]

    # Train final models for top combos and generate CSVs
    def train_final_predict(item_i, model_name):
        y = df_train[f'evgs_{item_i}'].values
        X_tr = np.nan_to_num(df_train[FEAT_COLS].values.astype(np.float32), nan=0.0)
        X_te = np.nan_to_num(df_test[FEAT_COLS].values.astype(np.float32), nan=0.0)
        spw = (len(y) - y.sum()) / max(y.sum(), 1)
        if model_name == 'XGBoost':
            clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                scale_pos_weight=spw, use_label_encoder=False, eval_metric='logloss',
                verbosity=0, random_state=42)
        elif model_name == 'LightGBM':
            clf = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                scale_pos_weight=spw, verbose=-1, random_state=42)
        else:
            clf = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
                auto_class_weights='Balanced', verbose=0, random_state=42)
        clf.fit(X_tr, y)
        return clf.predict(X_te)

    def make_submission(items_models, csv_name):
        m = df_base.copy()
        for item_i, model_name in items_models.items():
            preds = train_final_predict(item_i, model_name)
            for idx in range(len(df_test)):
                row = df_test.iloc[idx]
                pid = int(row['patient_id'])
                side_letter = 'L' if row['side'] == 'left' else 'R'
                col = f'{side_letter}{item_i}'
                row_mask = m['ID'] == f'track1-{pid}'
                m.loc[row_mask, col] = int(preds[idx])
        m.loc[t1_mask, 'Total'] = m.loc[t1_mask, item_cols_csv].sum(axis=1)
        # Track 2 rows preserved from base (overwritten by notebook 03)
        # Track-2 rows are preserved from the current base CSV.
        path = f'{DIAGNOSTIC_DIR}/{csv_name}'  # exploratory candidate; canonical R12 source is written by Cell 7b
        m.to_csv(path, index=False)
        n = sum((df_base.loc[t1_mask, c].values != m.loc[t1_mask, c].values).sum()
                for c in item_cols_csv)
        print(f'  diagnostic candidate {csv_name}: {n} source cells selected')

    # Generate top 5 combo CSVs
    print(f'\nGenerating top-5 combo CSVs:')
    for rank, (s1, items, models) in enumerate(unique_top[:5], 1):
        items_str = '_'.join(str(i) for i in sorted(items))
        csv_name = f'model_diversity_combo{rank}_items_{items_str}.csv'
        make_submission(models, csv_name)

    # Also generate the greedy result
    if current_switches:
        greedy_models = {i: best_model_per_item[i] for i in current_switches}
        items_str = '_'.join(str(i) for i in sorted(current_switches))
        csv_name = f'model_diversity_greedy_items_{items_str}.csv'
        make_submission(greedy_models, csv_name)

cell_end('Cell 7: Generate Submissions')


In [ ]:
# === Cell 7b: Emit canonical item-12 artifact ===
# Selection rule (codified in this notebook):
#   The canonical Track-1 source for EVGS item 12 is a single-item
#   model-diversity prediction trained with the model that won the
#   per-item LOSO-CV comparison (best_model_per_item[12]). The notebook
#   above also generates top-5 multi-item combos and a greedy combo for
#   exploratory review, but final assembly consumes only the R12 column
#   from this canonical file. Selecting the per-item-12 model keeps the
#   final CSV's R12 cell independent of which other items happen to be
#   bundled into a combo's CV score, which matches the provenance of the per-item-12
#   source slice released as the canonical artifact (R12 x 1 cell).
#
# This cell must run after Cell 7 in the same kernel session, since it
# reuses train_final_predict / make_submission and df_base. It fails fast
# (rather than emitting a stale canonical file) if item 12 has no entry in
# best_model_per_item.
# __canonical_item12_emit__
if 12 not in best_model_per_item:
    raise RuntimeError(
        'best_model_per_item[12] is missing. Cell 6 (LOSO model selection) '
        'must run successfully before this cell can emit the canonical item-12 source.')

if df_base is None:
    raise FileNotFoundError(
        'df_base is None; Cell 7 expects {SUBMIT_DIR}/reference_baseline.csv. '
        'Re-run reproduce.py from the start so the reference baseline is staged before this notebook.')

_canonical_chosen_model = best_model_per_item[12]
print(f'Canonical item-12 source: model={_canonical_chosen_model} for item 12 alone')
make_submission({12: _canonical_chosen_model}, 'item12_model_diversity.csv')
print(f'Wrote canonical artifact: {SUBMIT_DIR}/item12_model_diversity.csv')
print('Final assembly will consume R12 from this file.')

In [ ]:
# === Cell 8: Summary ===
cell_start('Summary')

total_time = sum(t for t in _cell_times.values() if t is not None)
time_report = '\n'.join(f'  {n}: {t:.1f}s ({t/60:.1f}min)'
    for n, t in _cell_times.items() if t is not None)

# Build greedy result string
greedy_items_str = str(sorted(current_switches)) if current_switches else 'none'
greedy_models_str = str({i: best_model_per_item[i] for i in sorted(current_switches)}) if current_switches else 'none'

print(f"""
######################################################################
#  model_diversity_item12 Model Diversity Selection — LOSO CV Space
#  #  Goal: Find optimal item-model combo to maximize S1 in CV
######################################################################

{'='*60}
Execution time:
{time_report}
  Total: {total_time:.1f}s ({total_time/60:.1f}min)

ML Baseline S1: {s1_baseline:.5f}

Greedy forward selection result:
  Items: {greedy_items_str}
  Models: {greedy_models_str}
  S1: {current_s1:.5f} (delta: {current_s1 - s1_baseline:+.5f})

Top 5 exhaustive combos:
""")

for rank, (s1, items, models) in enumerate(unique_top[:5], 1):
    print(f'  #{rank}: S1={s1:.5f} ({s1 - s1_baseline:+.5f}) items={list(items)} models={models}')

print(f"""
\nCSVs generated in: {SUBMIT_DIR}/model_diversity_item12_*.csv

Generated CSV artifacts for offline review:
  1. Greedy result (theoretically optimal in CV)
  2. Top exhaustive combos (smaller, may generalize better)

CV combo selection diagnostics — multi-criteria filtering used:
  - Small number of item switches (<= 6)
  - Delta >= 0.005 in CV
  - Items appearing in multiple top combos
{'='*60}
""")

cell_end('Summary')
